# Problem Set 5: Telicity, clustering, and factorization

*Fixed classes and variation across judgments*


## What remains fixed across judgments?

If a verb sense has a fixed telicity, repeated judgments need not be unanimous: a diagnostic can be noisy. But variation in the underlying construal makes a different prediction about repeated judgments. This assignment turns that distinction into models that you will derive, complete in Stan, and compare on responses from held out participants.

Read White, Grimm, and Glass's (2026) *Aspect is not a fixed property of linguistic expressions*, using the PDF shared on Zulip. The [public repository](https://github.com/SuperMereo/aspect-is-not-fixed) contains the data and analysis code. The paper distinguishes variation across verbs, across senses, and within senses. Our models isolate where a binary telicity class is shared. They simplify the paper's analyses, which also address durativity, argument properties, sense ambiguity, and event similarity.

Exercise 1 defines the observations and partitions participants. Exercises 2–3 construct the fixed and varying models. Exercise 4 fits them and checks predictions; Exercise 5 compares their predictive performance. Exercise 6 connects the models to clustering and matrix factorization and explains what their comparison can establish. All six exercises are required and sum to 100 points.

#### Assignment files
[Download Problem Set 5](https://aaronstevenwhite.io/stats-in-ling/downloads/ps5-assignment.zip). Extract the package, open [ps5.ipynb](ps5.ipynb), and select the R kernel. Keep the notebook in the extracted directory. See [README.md](README.md) for setup and [data/README.md](data/README.md) for measurements and attribution.

Complete the equations in **Your LaTeX**, the R blanks, and the numbered blanks in [fixed-telicity.stan](models/fixed-telicity.stan) and [varying-telicity.stan](models/varying-telicity.stan). Fill the Stan files before compiling them in Exercise 4. Report numerical results produced by visible code in **Your result**.

## Exercise 1: Distinguish judgments, sentences, senses, and verbs (10 points)

### 1.1 Read the two diagnostics

In Experiment 1a, participants completed an event description by choosing *in* or *for*, a numeral, and a time unit. The teaching table retains the preposition choice. A preference for *in* often diagnoses a telic construal, one in which an event has an inherent endpoint. An atelic construal does not require such a completion point; a preference for *for* often diagnoses it. Neither preposition is an infallible observation of telicity.

Experiment 1b helps explain this qualification. Participants saw an assigned preposition, numeral, and time unit and judged how much of that interval the event occupied. An *in* phrase can describe a delay before an event starts; a *for* phrase can describe the duration of a resulting state. Thus an *in* choice alone does not establish that the event itself filled the interval or culminated within it.

| Diagnostic block | Recorded categories | What is chosen or assigned? |
|---|---|---|
| Cloze | *for*, *in* | The participant chooses the preposition. |
| Overlap under *for* | 0, 1, 2, 3, 4 | The preposition is assigned; the participant chooses an overlap category. |
| Overlap under *in* | 0, 1, 2, 3, 4 | The preposition is assigned; the participant chooses an overlap category. |

Overlap 0 means none of the interval, 1 less than half, 2 about half, 3 more than half, and 4 the entire interval. Retain these as five categories. The models do not assume equal spacing or a cumulative-logit measurement rule. They also average over the omitted duration fields.

### 1.2 Use sentences with an annotated sense

Use the contentful sentences from the two experiments. Each has a PropBank roleset, a numbered verb-sense annotation, and a concrete or abstract reading. Their combination defines `sense_id` here. There are two sentences per annotated sense, allowing a comparison of descriptions whose arguments differ while that annotation stays fixed.

Bleached and templatic descriptions can be compatible with several senses; a single inherited label would conceal that uncertainty. They are excluded from this assignment. An annotated sense remains a proxy for an intended reading, rather than proof that every participant selected exactly that reading.


In [ ]:
library(dplyr)
library(ggplot2)
library(cmdstanr)
library(posterior)
source("scripts/helpers.R")
responses <- read.csv("data/aspect-responses.csv")
sentences <- read.csv("data/aspect-sentences.csv")
stopifnot(nrow(responses) == 4560, nrow(sentences) == 166,
          !anyDuplicated(sentences$sentence_id),
          !anyDuplicated(responses[c("participant", "sentence_id")]),
          n_distinct(sentences$verb) == 40,
          n_distinct(sentences$sense_id) == 83,
          all(table(sentences$sense_id) == 2))
table(responses$block)
sentences |> select(sentence_id, sentence, verb, sense_id, roleset, concreteness) |> head()


Identify the observation and each repeated unit. Explain why sentences are nested within annotated senses and verbs, while participants cross sentences. State what would be lost by treating a bleached description as an unambiguous sense observation.

### 1.3 Split participants before counting responses

The target is a new participant judging these recorded descriptions. A random trial split could put the same person's response tendencies in training and test data. Follow the [grouped cross-validation notes](https://aaronstevenwhite.io/stats-in-ling/model-criticism-and-prediction/grouped-cross-validation.html): assign each participant once, retaining all that person's trials in one partition.


In [ ]:
set.seed(581)
people <- sort(unique(responses$participant))
partition <- sample(rep(c("train", "train", "train", "validation", "test"),
                        length.out = length(people)))
split <- data.frame(participant = people, partition = partition)
responses <- left_join(responses, split, by = "participant")
train_rows <- filter(responses, partition == "train")
validation_rows <- filter(responses, partition == "validation")
test_rows <- filter(responses, partition == "test")
train_counts <- counts_for(___, sentences)
validation_counts <- counts_for(validation_rows, sentences)
test_counts <- counts_for(test_rows, sentences)
table(responses$block, responses$partition)
table(split$partition)
stopifnot(!any(train_rows$participant %in% validation_rows$participant),
          !any(train_rows$participant %in% test_rows$participant))
sapply(blocks, function(b) range(rowSums(train_counts[, b])))


`counts_for()` uses one row per sentence and twelve columns: two cloze categories, five overlap categories under *for*, and five under *in*. The three blocks have separate denominators. Category 0 is an observed overlap response; a block total of zero means that no response was observed.

Use training responses to inspect cloze preferences. Do not inspect test response rates before Exercise 5.


In [ ]:
training_cloze <- sentences |>
  mutate(n = rowSums(train_counts[, 1:2]),
         in_rate = train_counts[, 2] / n)
training_cloze |> filter(verb == "hit") |>
  select(sentence, sense_id, n, in_rate)
ggplot(filter(training_cloze, verb == "hit"), aes(sense_id, in_rate, group = sense_id)) +
  geom_line() + geom_point(aes(size = n)) +
  coord_flip() + labs(x = "Annotated sense", y = "Training cloze in rate")


Report partition sizes and any missing training blocks. Count rows by sentence and block across all partitions to determine whether a missing block is absent from the study or only from training; this does not require inspecting held out category rates. Describe variation between the two descriptions of an annotated sense. Explain why differing rates do not by themselves distinguish noisy measurement from differing construals.

#### Your result
Replace this sentence with your answers to Exercise 1.

## Exercise 2: Construct a model with a shared telicity class (20 points)

### 2.1 Separate the latent class from the response

Let $T=0$ denote an atelic class and $T=1$ a telic class. The class is unobserved. The observation is a response to one diagnostic, not a direct report of $T$.

We need a measurement model before response disagreement can tell us anything about class disagreement. Use a common cloze error probability $\epsilon=.10$:

| Class | Cloze *for* probability | Cloze *in* probability |
|---|---:|---:|
| $T=0$ | $1-\epsilon$ | $\epsilon$ |
| $T=1$ | $\epsilon$ | $1-\epsilon$ |

A telic description can thus receive a *for* response. With ten cloze judgments, it need not receive ten *in* responses. This error term can include ambiguity in the diagnostic as well as response noise; it is not an estimated rate of participant mistakes. The experiment did not independently measure $\epsilon$. Exercise 5 repeats the comparison with $\epsilon=.25$.

For each state $z$ and overlap block $d$, estimate a vector $\boldsymbol\theta_{zd}$ of five category probabilities. There are two latent states but five possible overlap responses; the class label and the response category are different variables. Do not stipulate that a telic state must always produce a particular overlap response. Put independent Dirichlet$(2,2,2,2,2)$ priors on these four vectors. This distribution assigns positive probabilities that sum to one, with prior mean $1/5$ per category and lower density near zero than a uniform Dirichlet$(1,\ldots,1)$ prior.

Write $\theta_{zdc}$ for the category probability in any block. The cloze entries are fixed by the table; the overlap entries are estimated. All four candidate models use these same measurement assumptions and overlap priors.

### 2.2 Decide which judgments share the class

Compare three fixed models:

| Model | Draw one class for each… | Judgments reusing that class |
|---|---|---|
| Fixed verb | verb $v$ | All descriptions and senses of that verb |
| Fixed sense | annotated sense $g$ | Both contentful descriptions of that sense |
| Fixed sentence | description $s$ | All judgments of that description |

The fixed sentence model matters: rejection of a verb or sense grouping could reflect differences between descriptions, without showing variation across repeated judgments of one description.

Use $g$ for a group under whichever fixed model is being fitted. Draw $T_g\sim\operatorname{Bernoulli}(.5)$ once for each group, independently across groups. Conditional on $T_g=z$, response $i$ in block $d[i]$ follows a categorical distribution:

$$
Y_i\mid T_{g[i]}=z,d[i]=d,\theta
\sim\operatorname{Categorical}(\boldsymbol\theta_{zd}).
$$

Review [categorical distributions](https://aaronstevenwhite.io/stats-in-ling/random-variables-and-distributions/bernoulli-and-categorical-distributions.html). Responses are independent conditional on the shared classes and measurement parameters. The class does not change when a new participant judges the group. The prior gives the two possible classes equal probability; that does not require equal numbers of telic and atelic groups after observing responses.

### 2.3 Derive the likelihood by adding over the shared state

Read [finite mixtures](https://aaronstevenwhite.io/stats-in-ling/latent-structure/gaussian-mixtures.html) and [Stan's treatment of discrete mixture states](https://mc-stan.org/docs/stan-users-guide/finite-mixtures.html). Stan samples the continuous probability parameters after the discrete class has been summed out.

Let $N_{gdc}$ be the training count in group $g$, block $d$, category $c$. Work through these steps in **Your LaTeX**.

1. Conditional on a state $z$, multiply the response probabilities. Repeated categories contribute powers:
   $f_{gz}=\prod_{dc}\theta_{zdc}^{N_{gdc}}$.
2. Take logs to obtain $\ell_{gz}=\sum_{dc}N_{gdc}\log\theta_{zdc}$.
3. Multiply each state's likelihood by its prior probability, then add the two alternatives:

   $$
   L_g=\text{enter fixed unit likelihood}.
   $$

4. Define $a_{gz}=\log(.5)+\ell_{gz}$. Rewrite the log likelihood as
   $\log L_g=\log(e^{a_{g0}}+e^{a_{g1}})$.
   Stan's `log_sum_exp()` evaluates this sum without directly exponentiating very small likelihoods.
5. Add $\log L_g$ over groups. The sum over states occurs **after multiplying all judgments sharing a class**. Moving it inside the product would let those judgments select different states.

The counts are a shortcut for calculating the likelihood of the original categorical trials. We omit multinomial coefficients throughout; we do not redefine each grouping's count table as a different observed dataset.

### 2.4 Recover uncertainty about the one shared class

Use [Bayes' rule](https://aaronstevenwhite.io/stats-in-ling/foundations/bayes-rule.html) and the [posterior membership notes](https://aaronstevenwhite.io/stats-in-ling/latent-structure/posterior-membership.html) to find $r_g=\mathbb{P}(T_g=1\mid\mathbf N_g,\theta)$. Start with the state 1 joint contribution, $.5e^{\ell_{g1}}$, and divide by the sum of both joint contributions:

$$
r_g=\text{enter fixed state probability}.
$$

For an arithmetic check, consider three *in* and one *for* cloze responses, with no overlap observations. At $\epsilon=.1$, the state 1 and state 0 likelihoods are $.9^3(.1)$ and $.1^3(.9)$. Their ratio is 81, giving $r_g=81/82$. This is uncertainty about one stable class; it is not a probability that a fresh class is chosen on each judgment.

### 2.5 Complete the shared-state Stan template

Open [models/fixed-telicity.stan](models/fixed-telicity.stan). `theta_for[k]` and `theta_in[k]` are five-category overlap distributions under the assigned prepositions, not cloze choice probabilities. `simplex[5]` declares one probability vector constrained to sum to one; `array[2]` creates one for each state. The matrix `response_prob` joins the fixed cloze pair and those two overlap vectors into twelve columns.

Its five labeled tasks follow the derivation:

1. **F1–F2:** enter the two cloze *in* probabilities from the measurement table.
2. **F3:** add the log probability of the observed count to the selected state's `lp[k]`. Stan state indices 1 and 2 represent mathematical states 0 and 1. Column `j` indexes a response category within one of the three blocks.
3. **F4:** add the marginalized group likelihood to `target`. It is added once per group, outside the loop over states.
4. **F5:** calculate the state 1 conditional probability in `generated quantities`, using $e^{a_{g1}-\log L_g}$. F3 also appears in that block; enter the same count contribution there.

`target` accumulates the log posterior density up to a constant. The Dirichlet statements add parameter priors; F4 adds observed-data likelihood. `generated quantities` computes summaries after a parameter draw and does not change that density. No discrete parameter appears in the `parameters` block.

Explain why putting `target += ...` inside the state loop would encode a different model. Also explain why a posterior class probability near one is compatible with occasional contrary cloze responses.

#### Your result
Replace this sentence with your answers to Exercise 2.

## Exercise 3: Allow the class to vary across judgments (20 points)

### 3.1 Give each sentence a construal probability

Different judgments of one description may use different aspectual construals. This could reflect differences among people as well as variation across occasions; these data do not separately identify those sources. Let $w_s$ be the probability of the telic construal for a judgment of sentence $s$. Put independent $w_s\sim\operatorname{Beta}(1,1)$ priors on the sentence weights.

Each judgment now draws its own state:

$$
T_i\mid w_{s[i]}\sim\operatorname{Bernoulli}(w_{s[i]}),
\qquad
Y_i\mid T_i=z,d[i]=d,\theta
\sim\operatorname{Categorical}(\boldsymbol\theta_{zd}).
$$

The measurement probabilities remain those in Exercise 2. A new class per judgment adds variation to the construal; the common error probability still allows variation in the response conditional on that class. The model thus represents both sources, though identifying them depends on the stated measurement restriction.

### 3.2 Derive one response probability before multiplying responses

Complete the following steps in **Your LaTeX**.

1. For category $c$ in block $d$, the atelic route has probability $(1-w_s)\theta_{0dc}$ and the telic route has probability $w_s\theta_{1dc}$.
2. Add those mutually exclusive routes:

   $$
   q_{sdc}=\text{enter varying response probability}.
   $$

3. Different judgments draw states independently conditional on $w$ and $\theta$. Multiply their marginal probabilities, then take logs and collect counts:

   $$
   \ell(w,\theta)=\text{enter varying log likelihood}.
   $$

4. Substitute the two cloze *in* probabilities into $q_{s,\mathrm{cloze},\textit{in}}$. Expand $(1-w_s)\epsilon+w_s(1-\epsilon)$ and collect $w_s$:

   $$
   q_{s,\mathrm{cloze},\textit{in}}=\text{enter cloze probability}.
   $$

At $\epsilon=.1$, weights $0,.25,.5,.75,1$ give cloze *in* probabilities $.1,.3,.5,.7,.9$. This continuous range contrasts with the two conditional probabilities available to a fixed class. Fixed-class posterior predictions can still lie between the endpoints because the one class is uncertain.

### 3.3 Translate the response mixture into Stan

Open [models/varying-telicity.stan](models/varying-telicity.stan). Its bounded vector `w` declares one probability per sentence. The weights need not sum to one across sentences; within a sentence, the two state probabilities $(1-w_s,w_s)$ sum to one.

1. **V1:** use `log_mix(w, log_p_1, log_p_0)` to compute the log probability of **one response**. Its first log probability is the telic component because `w` weights that component.
2. **V2:** multiply that log probability by the observed count and add it to `target`. The sum over states is now inside the response likelihood, rather than outside all responses of a group.
3. **V3:** calculate the marginal cloze *in* probability derived above.

The same $w_s$ is used across diagnostics for a sentence. This assumes one common construal propensity, while diagnostic and preposition effects enter through the response profiles. Discuss what this assumption would miss if adding an *in* or *for* modifier changed the propensity itself.

### 3.4 Identify the measurement restriction

For cloze alone, the observable probability is $p_s=\epsilon+(1-2\epsilon)w_s$. If both $\epsilon$ and $w_s$ were unrestricted, different pairs could give the same $p_s$. For instance, $(\epsilon,w_s)=(.1,.25)$ and $(.25,.1)$ both give $p_s=.3$.

Explain why response variation alone cannot establish how much is measurement error and how much is construal variation. State which assumption fixes the cloze measurement scale, and why testing a second error rate is useful. A Beta prior has no point mass at $w_s=0$ or 1; an interval inside $(0,1)$ is thus not a test rejecting a fixed class. Use predictive comparison instead.

#### Your result
Replace this sentence with your answers to Exercise 3.

## Exercise 4: Fit the models and check the measurement predictions (15 points)

### 4.1 Compile the completed templates and fit training responses

Use [CmdStanR](https://mc-stan.org/cmdstanr/articles/cmdstanr.html) with the CmdStan toolchain used in the earlier assignments. `compile_template()` checks that no blanks remain, writes a program under `stan_cache/`, and compiles it. `sample_cached()` fingerprints the code, data, seed, and sampler settings before reusing a saved fit.


In [ ]:
fixed_program <- compile_template("models/fixed-telicity.stan")
varying_program <- compile_template("models/varying-telicity.stan")
model_specs <- data.frame(
  model = c("fixed_verb", "fixed_sense", "fixed_sentence", "varying_judgment"),
  kind = c("fixed", "fixed", "fixed", "varying"),
  unit = c("verb", "sense_id", "sentence_id", "sentence_id"),
  seed = 582:585
)
fit_models <- function(counts, epsilon, label, seed_offset = 0) {
  stopifnot(epsilon > 0, epsilon < .5)
  result <- lapply(seq_len(nrow(model_specs)), function(i) {
    spec <- model_specs[i, ]
    group <- group_counts(counts, sentences[[spec$unit]])
    if (spec$kind == "fixed") {
      data <- list(G = nrow(group$counts), n = unname(group$counts), epsilon = epsilon)
      program <- fixed_program
    } else {
      data <- list(S = nrow(counts), n = unname(counts), epsilon = epsilon)
      program <- varying_program
    }
    fit <- sample_cached(program, data, paste(label, spec$model, sep = "-"),
                         seed = spec$seed + seed_offset)
    list(fit = fit, kind = spec$kind, group = group)
  })
  setNames(result, model_specs$model)
}
fits <- fit_models(train_counts, epsilon = .1, label = "train-e10")
diagnostics <- bind_rows(lapply(fits, check_fit), .id = "model")
diagnostics


Each fit uses four chains with 1,000 warmup and 1,000 retained iterations per chain, target acceptance .95, and maximum treedepth 12. `check_fit()` evaluates R-hat, bulk and tail effective sample size for the estimated probability parameters, divergences, treedepth hits, and E-BFMI. E-BFMI checks exploration across posterior energy levels; the [Stan diagnostics guide](https://mc-stan.org/learn-stan/diagnostics-warnings.html) explains the .3 threshold used here. The helper excludes fixed cloze constants and deterministic class probabilities that may round to exactly zero or one.

Report the checks. If one fails, inspect the chains and resolve the problem before interpreting the fit. The cache key changes when you change sampling settings. Successful sampling assesses computation under the model; it does not validate the measurement assumptions or independence of responses.

### 4.2 Distinguish estimated quantities from imposed differences


In [ ]:
weight_draws <- unclass(as_draws_matrix(fits$varying_judgment$fit$draws("w")))
weight_intervals <- apply(weight_draws, 2, quantile, c(.025, .5, .975))
weights <- cbind(sentences,
                 lower = weight_intervals[1, ], median = weight_intervals[2, ],
                 upper = weight_intervals[3, ])
weights |> arrange(sense_id) |> head(12)
pair <- match(c("The arrow hit the target.", "The hurricane hit the coastline."),
              sentences$sentence)
stopifnot(!anyNA(pair), n_distinct(sentences$sense_id[pair]) == 1)
weights[pair, ]
quantile(weight_draws[, pair[1]] - weight_draws[, pair[2]], c(.025, .5, .975))
draws <- state_draws(fits$varying_judgment)
apply(draws$theta, c(2, 3), median)


Describe the overlap profiles and the weight uncertainty for the two *hit* descriptions. Their difference is calculated within each posterior draw; use its interval rather than treating overlap between two separate intervals as a test. Which difference between the cloze profiles was imposed by definition? Explain why it cannot be reported as an empirical finding. Describe what a fixed model's `p_telic[g]` and the varying model's `w[s]` each mean.

### 4.3 Check sentence-level cloze predictions

An overall *in* rate can hide poor predictions for particular descriptions. Use the [posterior predictive check notes](https://aaronstevenwhite.io/stats-in-ling/statistical-inference/posterior-predictive-checks.html) to simulate repeated response counts using each fitted model, keeping the training totals fixed.


In [ ]:
cloze_checks <- bind_rows(lapply(fits, function(fit)
  cloze_check(fit, train_counts, sentences)), .id = "model")
cloze_checks |> group_by(model) |>
  summarize(checked = n(), outside = sum(outside), .groups = "drop")
ggplot(cloze_checks, aes(observed, median)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed") +
  geom_linerange(aes(ymin = lower, ymax = upper), alpha = .2) +
  geom_point(alpha = .5) + facet_wrap(~model) +
  labs(x = "Observed training in rate", y = "Replicated in rate")


The helper displays descriptions with at least four training cloze judgments. In the fixed models it draws **one state per group** in each replication and reuses it for the group's sentences and blocks. In the varying model it samples counts from the mixed response probabilities, integrating an independent state for each judgment.

Report which descriptions fall outside their ranges and describe whether a model tends to force intermediate observed preferences toward an endpoint. A model may have no observations outside these wide ranges; explain why that alone would not establish its adequacy. The ranges are pointwise, not a simultaneous test across all descriptions. These checks concern recorded sentences and the training sample.

#### Your result
Replace this sentence with your answers to Exercise 4.

## Exercise 5: Compare models on judgments from held out participants (20 points)

### 5.1 Compute the same predictive score for every model

Read the notes on [predictive loss](https://aaronstevenwhite.io/stats-in-ling/model-criticism-and-prediction/predictive-loss.html) and [separating training, validation, and test data](https://aaronstevenwhite.io/stats-in-ling/model-criticism-and-prediction/training-and-test-data.html). For a new judgment of a recorded sentence, average over the training posterior. In a fixed model, use its conditional probability $r_g$ of the one shared state; do not estimate a new class using the held out response. In the varying model, use the training posterior of $w_s$.

Within each parameter draw, form $(1-a_s)\theta_{0dc}+a_s\theta_{1dc}$, where $a_s$ is $r_{g[s]}$ or $w_s$. Then average over draws. `predict_probabilities()` follows this order; multiplying posterior mean weights and posterior mean profiles can give a different answer because they can be dependent.

Let $\widehat q_{sdc}$ denote that posterior predictive probability. Derive the average log score in **Your LaTeX**: one judgment contributes $\log\widehat q_{sdc}$; $N^{H}_{sdc}$ matching held out judgments contribute $N^{H}_{sdc}\log\widehat q_{sdc}$; sum and divide by their total:

$$
\overline\ell_H=\text{enter predictive log score}.
$$

This is a pointwise score for a single response, averaged over responses. It is not the joint probability of all responses sharing a fixed class. For cloze, also report Brier loss $(y-\widehat p)^2$, with *in* coded 1. Higher log score and lower Brier loss are better.


In [ ]:
validation_q <- lapply(fits, predict_probabilities)
validation_scores <- bind_rows(lapply(validation_q, function(q)
  as.data.frame(t(predictive_scores(validation_counts, q)))), .id = "model")
validation_scores
manual_validation_score <- sum(validation_counts * ___) / sum(validation_counts)
stopifnot(abs(manual_validation_score -
                validation_scores$overall[validation_scores$model == "varying_judgment"]) < 1e-12)
winner <- validation_scores$model[which.max(validation_scores$overall)]
winner


Compare the fixed sense and fixed sentence models before interpreting the varying model's gain. Why is beating only the fixed verb model insufficient to establish variation within a sense? Why is beating only the fixed sense model insufficient to establish variation within one description? A numerical maximum defines the selection rule even when the differences are too small to establish a substantive preference; distinguish the selected model from strong evidence for its assumptions.

### 5.2 Repeat the validation comparison with a different error rate


In [ ]:
sensitivity_fits <- fit_models(train_counts, epsilon = .25, label = "train-e25")
sensitivity_diagnostics <- bind_rows(lapply(sensitivity_fits, check_fit), .id = "model")
sensitivity_diagnostics
sensitivity_scores <- bind_rows(lapply(sensitivity_fits, function(fit)
  as.data.frame(t(predictive_scores(validation_counts, predict_probabilities(fit))))),
  .id = "model")
sensitivity_scores


Increasing $\epsilon$ makes disagreement more likely under a fixed state. It also changes the varying model: its allowed cloze probabilities shrink from $[.1,.9]$ to $[.25,.75]$, and averaging uncertain weights can pull its predictions further toward .5. Compare this restriction with the extreme training preferences before interpreting a change in ranking.

Report whether the ordering of the models changes, and whether the varying model's advantage grows or shrinks. Keep this as a declared sensitivity analysis: do not choose a new error rate after looking at test results. A favorable result is conditional on the common measurement model, not a test of every possible fixed-telicity theory.

### 5.3 Refit with training and validation responses, then evaluate test once

Retain $\epsilon=.10$ for the primary test comparison. The four specifications and the validation selection rule were declared in advance. Fit them to combined training and validation responses and report test scores for all four, identifying which was selected by validation. Do not change that selection using the test results.


In [ ]:
development_counts <- train_counts + validation_counts
final_fits <- fit_models(development_counts, epsilon = .1, label = "development-e10",
                        seed_offset = 100)
final_diagnostics <- bind_rows(lapply(final_fits, check_fit), .id = "model")
final_diagnostics
test_q <- lapply(final_fits, predict_probabilities)
test_scores <- bind_rows(lapply(test_q, function(q)
  as.data.frame(t(predictive_scores(test_counts, q)))), .id = "model")
test_scores
test_scores |> filter(model == winner)
gain_draws <- bootstrap_gain(test_rows, test_q$fixed_sentence,
                            test_q$varying_judgment, sentences)
quantile(gain_draws, c(.025, .5, .975))


The paired bootstrap resamples whole held out participants. Positive gain favors varying judgments over a fixed sentence class. Describe its distribution and its limits: predictions stay fixed, so the resampling does not refit the models or include every source of estimation uncertainty.

Refitting can change predictions differently in the two models. A fixed model can become more certain about its one class and move its predictive mean toward an endpoint. A varying model can instead become more certain about an intermediate weight. Use this distinction when interpreting validation and test results.

Report overall and block-specific scores. Do the models differ primarily in cloze, overlap, or both? The partitions can contain different proportions of the diagnostic tasks, so interpret the overall score alongside each block's score.

The likelihood omits participant effects and duration predictors. A participant split prevents the same person's rows appearing on both sides, but it does not repair that likelihood or establish calibrated population uncertainty. An ordinary chi-squared likelihood ratio test is not an appropriate substitute for this predictive comparison of discrete mixtures and boundary restrictions.

#### Your result
Replace this sentence with your answers to Exercise 5.

## Exercise 6: Connect shared classes, factorization, and interpretation (15 points)

### 6.1 Show why equal mean predictions need not imply the same model

Consider two future cloze judgments, from different participants, of the same description. Hold the response parameters fixed, use $\epsilon=.1$, and compare a fixed class with probability $r$ of being telic to varying judgments with weight $w=r$.

Complete these steps in **Your LaTeX**.

1. In the fixed model, both judgments use the same state. Conditional independence gives $\epsilon^2$ if it is atelic and $(1-\epsilon)^2$ if it is telic. Average over that one uncertain state:

   $$
   \mathbb{P}(Y_1=\textit{in},Y_2=\textit{in}\mid\mathbf N,\theta)
   =\text{enter two fixed judgments}.
   $$

2. In the varying model, draw states independently for the two judgments, conditional on $w$. Multiply their marginal probabilities:

   $$
   \mathbb{P}(Y_1=\textit{in},Y_2=\textit{in}\mid w,\theta)
   =\text{enter two varying judgments}.
   $$

3. Set $w=r$ and abbreviate $1-2\epsilon$ by $\delta$. The fixed expression simplifies to $\epsilon^2+r\delta$; the varying expression expands to $\epsilon^2+2\epsilon r\delta+r^2\delta^2$. Subtract, cancel $\epsilon^2$, and use $1-2\epsilon=\delta$ to factor:

   $$
   \mathbb{P}_{\mathrm{fixed}}(\textit{in},\textit{in})
   -\mathbb{P}_{\mathrm{varying}}(\textit{in},\textit{in})
   =\text{enter two judgment difference}.
   $$

4. Check with $r=w=.5$. Both models give marginal *in* probability .5; the joint probabilities are .41 and .25. Explain why fractional posterior membership of a fixed class is not itself evidence of changing classes.

This calculation conditions on the continuous parameters. Integrating their posterior uncertainty can also create dependence, including in the varying model; its joint posterior prediction is not generally the product of two posterior mean probabilities.

### 6.2 Derive the factorization and its geometric restriction

For one parameter draw, stack each state's twelve response probabilities into rows of a $2\times12$ matrix $\Theta$. Put $(1-w_s,w_s)$ in row $s$ of $W$. Multiplication gives the varying model's $S\times12$ probability matrix:

$$
Q=W\Theta
=\mathbf1\boldsymbol\theta_0^\top+
 \mathbf w(\boldsymbol\theta_1-\boldsymbol\theta_0)^\top.
$$

A row is a weighted average of two response profiles. Thus its possible positions form a line segment between those profiles. A known fixed class instead selects one endpoint. Posterior uncertainty about that fixed class can average the endpoints without changing the assumption that the class itself is shared.

Read [centering](https://aaronstevenwhite.io/stats-in-ling/factorization/centering-and-scaling.html), [SVD](https://aaronstevenwhite.io/stats-in-ling/factorization/singular-value-decomposition.html), and [scores and loadings](https://aaronstevenwhite.io/stats-in-ling/factorization/scores-and-loadings.html). Subtract the column means of $Q$. The constant term cancels, leaving

$$
Q_c=(\mathbf w-\bar w\mathbf1)
(\boldsymbol\theta_1-\boldsymbol\theta_0)^\top.
$$

Explain why every column of $Q_c$ is a scalar multiple of the same centered weight vector, giving rank at most one. Check this on one parameter draw, then inspect the singular values of the observed, separately smoothed block proportions.


In [ ]:
one_draw <- state_draws(fits$varying_judgment, ndraws = 1)
w <- one_draw$weights[1, ]
W <- cbind(___, w)
Q <- W %*% one_draw$theta[1, , ]
Qc <- sweep(Q, 2, colMeans(Q))
svd(Qc)$d[1:5]

P <- train_counts + .5
for (b in blocks) P[, b] <- P[, b] / rowSums(P[, b])
Pc <- sweep(P, 2, colMeans(P))
singular_values <- svd(Pc)$d
singular_values[1:5]
c(first_share = singular_values[1]^2 / sum(singular_values^2))


The smoothing assigns uniform proportions to an unobserved block. It is a descriptive choice, not observed data; unequal block totals also produce different amounts of smoothing. SVD here summarizes a matrix, not another categorical generative model. The resulting matrix need not have rank one even if the probability model is true.

Check how much second-direction variation this model predicts from sampling. Use the same block totals and smoothing in posterior replications:


In [ ]:
spectrum_replications <- replicate_counts(fits$varying_judgment, train_counts,
                                         ndraws = 200, seed = 589)
replicated_second <- sapply(seq_len(dim(spectrum_replications)[1]), function(d) {
  rates <- spectrum_replications[d, , ] + .5
  for (b in blocks) rates[, b] <- rates[, b] / rowSums(rates[, b])
  svd(sweep(rates, 2, colMeans(rates)))$d[2]
})
c(observed_second = singular_values[2],
  quantile(replicated_second, c(.025, .5, .975)))


Compare the observed second singular value with this range before claiming an additional aspectual dimension. Excess variation could also reflect omitted participant or duration effects. Extra response profiles or separate telicity and durativity dimensions are possible extensions, but they would also need a held out predictive comparison.

The rank claim holds for one parameter draw. Averaging probability matrices over uncertain profiles and weights need not preserve rank one. Explain why the mean predictive matrix used in Exercise 5 should not be tested against that rank bound as if its parameters were fixed.

### 6.3 State what the comparison establishes

Write a paragraph of six to eight sentences connecting the modeling assumptions to the results. Distinguish fixed verbs, fixed annotated senses, fixed descriptions, and varying judgments. Explain which predictions were evaluated, how the measurement-error sensitivity affects the result, and why success on these recorded sentences does not establish predictions for unseen senses or verbs. Each participant contributed at most one judgment of a description; explain why this design cannot establish that the same person would switch construal on a repeated presentation.

For a new fixed group, the class prior is .5. For a completely new sentence in the varying model, its Beta$(1,1)$ prior has mean .5; this model does not borrow a sentence weight from its annotated sense. Explain how a hierarchy linking sentence weights to sense and verb could change that target. The paper's treatment of sense uncertainty and richer structural predictors addresses questions beyond these simplified models; do not present this comparison as a replication of every analysis in the paper.

#### Your result
Replace this sentence with your answers to Exercise 6.

Submit the completed notebook and both completed Stan templates. Keep data and helpers together while working. Report all six result blocks, equations, and visible code output; omit compiled programs, cached fits, and sampler CSV files from the submission.
